# 05 - Validation receipt and reproducibility

## Objective

Run the same public Case twice through the CLI, inspect the persisted evidence, and learn which identity fields are deterministic and which timestamps are not.

## Source, assumptions, and units

The source is the bundled IEEE13 public demonstrator selected by `cept study demo load-flow`. The Case fingerprint identifies the typed input; solver voltage magnitudes are pu, and the receipt identifies OpenDSS and the installed public version. No field measurements or project-specific settings are supplied.

## Prediction

Two runs with the same Case and solver should have the same Case fingerprint and load-flow payload, while attempt identity and creation timestamps may differ. Both exact run directories should verify with the public claim `WORKFLOW_VALIDATED`.

## Action

Stream two `cept study demo load-flow` commands to two explicit run paths, then stream `cept study verify` for each path. No `latest` directory or modification time is used to select evidence.

## Verification

Read `case.json`, `results.json`, and `public-verification.json` from both named runs. Compare the Case and load-flow payloads, then assert the exact verification receipts.

## Interpretation

A verification receipt proves the persisted public artifact set is internally consistent for its bounded workflow. It does not turn a demonstrator into field evidence, independent reference agreement, or `PROJECT_VALIDATED`.

## Exercise

Run the same cells after changing the exact output directory names, then change one Case input in a lesson that owns an inline Case. Predict which fingerprint and result fields should change before rerunning.

## Runtime requirements

Use Python 3.10 or newer with an existing installed `cept` command, or provide a caller-owned wheel through `CEPT_WHEEL_URL` and its exact `CEPT_WHEEL_SHA256`. The wheel must provide CEPT and OpenDSS. No released PyPI version is assumed. Jupyter is needed only to execute the notebook.

In [1]:
# @title Setup — run once, then read the results below
import urllib.request, hashlib
_HELPER_URL = "https://raw.githubusercontent.com/sarutesri/cept-studio-edu/75b4f394aa096a604123e6e1739d2581e8ef9476/public/notebooks/_lesson.py"
_HELPER_SHA256 = "8618face0c62b85127ffadc7b662bc177ab3ba5a36e32a09ee5223f562e02ba2"
_blob = urllib.request.urlopen(_HELPER_URL, timeout=60).read()
assert hashlib.sha256(_blob).hexdigest() == _HELPER_SHA256, "lesson helper hash mismatch"
exec(compile(_blob, "lesson helper", "exec"))


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version


cept-power-studio 0.2.0.dev0
lesson helpers ready: cli/read/table/cards + WORKSPACE.


### Run the identical Case twice

Two separate run directories, same demo inputs. Separate `attempt_id` values with identical fingerprints is the whole lesson.


In [2]:
RUN_ONE = WORKSPACE / 'runs' / '05-reproducibility-1'
RUN_TWO = WORKSPACE / 'runs' / '05-reproducibility-2'
first_summary = cli('study', 'demo', 'load-flow', '--network', 'ieee13', '--out', RUN_ONE, '--force')
second_summary = cli('study', 'demo', 'load-flow', '--network', 'ieee13', '--out', RUN_TWO, '--force')
first_verify = cli('study', 'verify', RUN_ONE)
second_verify = cli('study', 'verify', RUN_TWO)
assert first_summary['status'] == 'PASS' and second_summary['status'] == 'PASS'
assert first_verify['passed'] is True and second_verify['passed'] is True


$ cept study demo load-flow --network ieee13 --out '<notebook-workspace>\runs\05-reproducibility-1' --force


→ exit 0


$ cept study demo load-flow --network ieee13 --out '<notebook-workspace>\runs\05-reproducibility-2' --force


→ exit 0


$ cept study verify '<notebook-workspace>\runs\05-reproducibility-1'


→ exit 0


$ cept study verify '<notebook-workspace>\runs\05-reproducibility-2'


→ exit 0


### Receipts first, then the verdict

Fingerprints identify the persisted Case; the claim stays workflow-bounded either way.


In [3]:
case_one = read(RUN_ONE / 'case.json')
case_two = read(RUN_TWO / 'case.json')
result_one = read(RUN_ONE / 'results.json')
result_two = read(RUN_TWO / 'results.json')
receipt_one = read(RUN_ONE / 'public-verification.json')
receipt_two = read(RUN_TWO / 'public-verification.json')
table(['run', 'case fingerprint', 'study type', 'engine', 'claim', 'verified'], [(str(RUN_ONE), receipt_one['case_fingerprint'], receipt_one['study_type'], receipt_one['engine'], receipt_one['claim'], first_verify['passed']), (str(RUN_TWO), receipt_two['case_fingerprint'], receipt_two['study_type'], receipt_two['engine'], receipt_two['claim'], second_verify['passed'])])
assert receipt_one['claim'] == 'WORKFLOW_VALIDATED' and receipt_two['claim'] == 'WORKFLOW_VALIDATED'


| run | case fingerprint | study type | engine | claim | verified |
| --- | --- | --- | --- | --- | --- |
| <notebook-workspace>\runs\05-reproducibility-1 | 748c8026c9d6 | load_flow | opendss | WORKFLOW_VALIDATED | True |
| <notebook-workspace>\runs\05-reproducibility-2 | 748c8026c9d6 | load_flow | opendss | WORKFLOW_VALIDATED | True |


### Full voltage table - every bus must match

All buses from the first run, checked against the second run below. Timestamps and attempt IDs differ; engineering payloads must not.


In [4]:
table(['run', 'bus', 'phase', 'voltage magnitude', 'unit'], [(str(RUN_ONE), row['bus'], row['phase'], row['v_pu'], 'pu') for row in result_one['load_flow']['bus_voltages']])

compared_buses = len(result_one['load_flow']['bus_voltages'])
cards([
    ('Fingerprints match', str(result_one['case_fingerprint'] == result_two['case_fingerprint']), 'identical persisted Case'),
    ('Claim', receipt_one['claim'], 'workflow evidence only'),
    ('Buses compared', str(compared_buses), 'full voltage table below'),
], title='5 · Reproducibility receipt')
assert case_one == case_two
assert result_one['case_fingerprint'] == result_two['case_fingerprint']
assert result_one['load_flow'] == result_two['load_flow']
assert receipt_one['attempt_id'] != receipt_two['attempt_id']


| run | bus | phase | voltage magnitude | unit |
| --- | --- | --- | --- | --- |
| <notebook-workspace>\runs\05-reproducibility-1 | sourcebus | 1 | 0.999974 | pu |
| <notebook-workspace>\runs\05-reproducibility-1 | sourcebus | 2 | 0.999994 | pu |
| <notebook-workspace>\runs\05-reproducibility-1 | sourcebus | 3 | 0.99995 | pu |
| <notebook-workspace>\runs\05-reproducibility-1 | 650 | 1 | 0.999911 | pu |
| <notebook-workspace>\runs\05-reproducibility-1 | 650 | 2 | 0.999971 | pu |
| <notebook-workspace>\runs\05-reproducibility-1 | 650 | 3 | 0.999931 | pu |
| <notebook-workspace>\runs\05-reproducibility-1 | rg60 | 1 | 1.056033 | pu |
| <notebook-workspace>\runs\05-reproducibility-1 | rg60 | 2 | 1.037386 | pu |
| <notebook-workspace>\runs\05-reproducibility-1 | rg60 | 3 | 1.05605 | pu |
| <notebook-workspace>\runs\05-reproducibility-1 | 633 | 1 | 1.011301 | pu |
| <notebook-workspace>\runs\05-reproducibility-1 | 633 | 2 | 1.027017 | pu |
| <notebook-workspace>\runs\05-reproducibility-1 | 63

The comparison uses actual persisted Case and solver payloads. Attempt IDs and timestamps identify separate executions; they are not used to choose which result is authoritative. Keep both exact run paths and their public verification receipts when sharing this exercise.